# Prepare Hourly Data

Add two half-hour readings to make one hourly value. Use the four-home sample.

Open this notebook in `notebooks/M2/`. Run the cells in order. Saving again replaces this notebook's output files.


## Choose the project folder

In [11]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

WindowsPath('D:/Desktop/Data Mining Project')

## Read the sample

half-hour consumption readings for 4 homes

In [12]:
import pandas as pd
raw_path = PROJECT_ROOT / "data/pilot/half_hour_sample.csv"
raw = pd.read_csv(raw_path, dtype=str)
raw.head()

,LCLid,tstp,energy(kWh/hh)
0,MAC000414,2013-01-01 00:00:00.0000000,0.13
1,MAC000414,2013-01-01 00:30:00.0000000,0.125
2,MAC000414,2013-01-01 01:00:00.0000000,0.148
3,MAC000414,2013-01-01 01:30:00.0000000,0.095
4,MAC000414,2013-01-01 02:00:00.0000000,0.098


In [13]:
raw['LCLid'].unique() # 4 homes

array(['MAC000414', 'MAC000675', 'MAC001312', 'MAC005456'], dtype=object)

## Keep the four homes

In [15]:
homes = ["MAC000414", "MAC000675", "MAC001312", "MAC005456"]
raw = raw[raw["LCLid"].isin(homes)].copy()
raw["LCLid"].value_counts()

LCLid
MAC001312    17520
MAC000675    17517
MAC000414    17325
MAC005456    16894
Name: count, dtype: int64

## Read dates and energy

In [17]:
raw.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69256 entries, 0 to 69255
Data columns (total 3 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   LCLid           69256 non-null  object
 1   tstp            69256 non-null  object
 2   energy(kWh/hh)  69256 non-null  object
dtypes: object(3)
memory usage: 1.6+ MB


In [19]:
parsed = raw[["LCLid"]].copy()
parsed["ts"] = pd.to_datetime(raw["tstp"], utc=True) # convert dates to "datetime" datatype
parsed["energy"] = pd.to_numeric(raw["energy(kWh/hh)"].str.strip(), errors="coerce") # convert energy to numeric datatype
print(parsed.info())
parsed.head()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 69256 entries, 0 to 69255
Data columns (total 3 columns):
 #   Column  Non-Null Count  Dtype              
---  ------  --------------  -----              
 0   LCLid   69256 non-null  object             
 1   ts      69256 non-null  datetime64[ns, UTC]
 2   energy  69256 non-null  float64            
dtypes: datetime64[ns, UTC](1), float64(1), object(1)
memory usage: 1.6+ MB
None


,LCLid,ts,energy
0,MAC000414,2013-01-01 00:00:00+00:00,0.130
1,MAC000414,2013-01-01 00:30:00+00:00,0.125
2,MAC000414,2013-01-01 01:00:00+00:00,0.148
3,MAC000414,2013-01-01 01:30:00+00:00,0.095
4,MAC000414,2013-01-01 02:00:00+00:00,0.098


## Find the start of each half-hour

Raw times mark half-hour ends. Subtract 30 minutes before grouping.

In [20]:
interval_start = parsed["ts"] - pd.Timedelta(minutes=30)
parsed["hour_start"] = interval_start.dt.floor("h")
parsed.head()

,LCLid,ts,energy,hour_start
0,MAC000414,2013-01-01 00:00:00+00:00,0.130,2012-12-31 23:00:00+00:00
1,MAC000414,2013-01-01 00:30:00+00:00,0.125,2013-01-01 00:00:00+00:00
2,MAC000414,2013-01-01 01:00:00+00:00,0.148,2013-01-01 00:00:00+00:00
3,MAC000414,2013-01-01 01:30:00+00:00,0.095,2013-01-01 01:00:00+00:00
4,MAC000414,2013-01-01 02:00:00+00:00,0.098,2013-01-01 01:00:00+00:00


## Add readings for each hour

Keep the sum when both readings are present. Otherwise, leave it missing.

In [ ]:
hourly = parsed.groupby(["LCLid", "hour_start"])["energy"].agg(n_valid="count", kWh_sum="sum")
hourly["kWh"] = hourly["kWh_sum"].where(hourly["n_valid"] == 2).round(3)
hourly.head()

## List all hours in 2013

In [ ]:
hours = pd.date_range("2013-01-01 00:00", "2013-12-31 23:00", freq="h", tz="UTC")
grid = pd.MultiIndex.from_product([sorted(homes), hours], names=["LCLid", "hour_start"])
len(grid)

## Keep missing hours in the table

In [ ]:
hourly = hourly.reindex(grid)
hourly["n_valid"] = hourly["n_valid"].fillna(0).astype(int)
hourly["is_complete"] = hourly["n_valid"] == 2
hourly = hourly[["kWh", "n_valid", "is_complete"]].reset_index()
hourly.head()

## Count complete and missing hours

In [ ]:
hourly["is_complete"].value_counts()

## Choose one hour to check

In [ ]:
example_home = "MAC001312"
example_hour = pd.Timestamp("2013-01-01 00:00", tz="UTC")
example_end = example_hour + pd.Timedelta(hours=1)
example_home, example_hour

## Show its two half-hour readings

In [ ]:
home_rows = parsed[parsed["LCLid"] == example_home]
raw_example = home_rows[
    (home_rows["ts"] > example_hour) & (home_rows["ts"] <= example_end)
]
raw_example[["LCLid", "ts", "energy"]]

## Show the saved hourly value

In [ ]:
hourly_example = hourly[
    (hourly["LCLid"] == example_home) & (hourly["hour_start"] == example_hour)
]
hourly_example

## Compare the sum

In [ ]:
manual_sum = round(raw_example["energy"].sum(), 3)
saved_sum = hourly_example["kWh"].iloc[0]
manual_sum, saved_sum

## Choose the output folder

In [ ]:
output_dir = PROJECT_ROOT / "data/pilot"
output_dir.mkdir(parents=True, exist_ok=True)
output_path = output_dir / "hourly_sample.csv"
output_path

## Save the hourly file

In [ ]:
hourly.to_csv(output_path, index=False)
hourly.head()